<a href="https://colab.research.google.com/github/nsalh6831-debug/IR-Search-Engine/blob/main/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!pip install ir-datasets nltk rank-bm25 \
    sentence-transformers scikit-learn \
    pandas numpy tqdm

In [37]:
import ir_datasets
import nltk
import pandas as pd
import numpy as np
from tqdm import tqdm

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tabulator')

print(" All libraries imported successfully!")

 All libraries imported successfully!


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Error loading punkt_tabulator: Package 'punkt_tabulator'
[nltk_data]     not found in index


In [38]:
# Cell 3: Load MS MARCO Dataset
import ir_datasets

print("Loading MS MARCO...")
dataset = ir_datasets.load("msmarco-passage/train")

count = 0
for doc in dataset.docs_iter():
    print(f"ID: {doc.doc_id}")
    print(f"Text: {doc.text[:100]}...")
    print("---")
    count += 1
    if count == 3:
        break

print("MS MARCO loaded successfully!")

Loading MS MARCO...
ID: 0
Text: The presence of communication amid scientific minds was equally important to the success of the Manh...
---
ID: 1
Text: The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peacefu...
---
ID: 2
Text: Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an...
---
MS MARCO loaded successfully!


In [39]:

from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/IR_Project', exist_ok=True)
os.makedirs('/content/drive/MyDrive/IR_Project/msmarco', exist_ok=True)
os.makedirs('/content/drive/MyDrive/IR_Project/trec_covid', exist_ok=True)

print(" Google Drive connected!")
print(" Project folders created!")

Mounted at /content/drive
 Google Drive connected!
 Project folders created!


In [40]:
#
import shutil
import os

print("Saving MS MARCO to Drive...")

shutil.copytree(
    '/root/.ir_datasets',
    '/content/drive/MyDrive/IR_Project/ir_datasets_cache',
    dirs_exist_ok=True
)

print(" MS MARCO saved to Drive successfully!")

Saving MS MARCO to Drive...
 MS MARCO saved to Drive successfully!


In [41]:
# Cell 5: Load TREC-COVID Dataset
print("Loading TREC-COVID...")
dataset_covid = ir_datasets.load("cord19/trec-covid")

count = 0
for doc in dataset_covid.docs_iter():
    print(f"ID: {doc.doc_id}")
    print(f"Text: {doc.title[:100]}...")
    print("---")
    count += 1
    if count == 3:
        break

print("TREC-COVID loaded successfully!")

[INFO] [starting] building docstore


Loading TREC-COVID...


[INFO] If you have a local copy of https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/80d664e496b8b7e50a39c6f6bb92e0ef
[INFO] [starting] https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv
docs_iter:   0%|                                    | 0/192509 [00:00<?, ?doc/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 0.00/269M [00:00<?, ?B/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 24.6k/269M [00:00<22:13, 202kB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 57.3k/269M [00:00<20:16, 221kB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.1%| 139k/269M [00:00<12:43, 353kB/s] 
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07

ID: ug7v899j
Text: Clinical features of culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Ho...
---
ID: 02tnwd4m
Text: Nitric oxide: a pro-inflammatory mediator in lung disease?...
---
ID: ejv2xln0
Text: Surfactant protein-D and pulmonary host defense...
---
TREC-COVID loaded successfully!


In [42]:
#
import shutil
shutil.copytree(
    '/root/.ir_datasets',
    '/content/drive/MyDrive/IR_Project/ir_datasets_cache',
    dirs_exist_ok=True
)
print(" TREC-COVID saved to Drive!")

 TREC-COVID saved to Drive!


In [43]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt_tab', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords + Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return tokens

#
sample = "The Manhattan Project was a research project during World War II"
result = preprocess_text(sample)
print("Before:", sample)
print("After:", result)
print("✅ Preprocessing function ready!")

Before: The Manhattan Project was a research project during World War II
After: ['manhattan', 'project', 'research', 'project', 'world', 'war']
✅ Preprocessing function ready!


In [44]:
# Cell: Preprocess MS MARCO samples
from tqdm import tqdm

print("Processing MS MARCO...")
dataset = ir_datasets.load("msmarco-passage/train")

msmarco_processed = []
count = 0

for doc in tqdm(dataset.docs_iter()):
    processed = preprocess_text(doc.text)
    msmarco_processed.append({
        'doc_id': doc.doc_id,
        'original': doc.text[:100],
        'processed': processed
    })
    count += 1
    if count == 1000:
        break

print(f"✅ Processed {len(msmarco_processed)} documents!")
print("Example:")
print(msmarco_processed[0])

Processing MS MARCO...


999it [00:00, 2506.11it/s]

✅ Processed 1000 documents!
Example:
{'doc_id': '0', 'original': 'The presence of communication amid scientific minds was equally important to the success of the Manh', 'processed': ['presence', 'communication', 'amid', 'scientific', 'mind', 'equally', 'important', 'success', 'manhattan', 'project', 'scientific', 'intellect', 'cloud', 'hanging', 'impressive', 'achievement', 'atomic', 'researcher', 'engineer', 'success', 'truly', 'meant', 'hundred', 'thousand', 'innocent', 'life', 'obliterated']}


In [45]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('punkt_tab', quiet=True)

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    tokens = [t for t in tokens
              if t not in stop_words and len(t) > 2]
    # 5. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    # 6. Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

# تجربة
sample = "The Manhattan Project was a research project during World War II"
result = preprocess_text(sample)
print("Before:", sample)
print("After:", result)
print("✅ Preprocessing with Stemming ready!")

Before: The Manhattan Project was a research project during World War II
After: ['manhattan', 'project', 'research', 'project', 'world', 'war']
✅ Preprocessing with Stemming ready!


In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [47]:
documents = []

for doc in msmarco_processed:
    documents.append(
        " ".join(doc["processed"])
    )

print(documents[0])

presence communication amid scientific mind equally important success manhattan project scientific intellect cloud hanging impressive achievement atomic researcher engineer success truly meant hundred thousand innocent life obliterated


In [48]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    documents
)

print(tfidf_matrix.shape)

(1000, 7190)


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
def tfidf_search(query):

    query_processed = preprocess_text(query)

    query_text = " ".join(query_processed)

    query_vector = vectorizer.transform(
        [query_text]
    )

    similarities = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    top_indices = similarities.argsort()[-5:][::-1]

    return top_indices

In [51]:
results = tfidf_search(
    "manhattan project"
)

print(results)

[2 7 3 8 1]


In [52]:
for idx in results:
    print("DOC ID:", msmarco_processed[idx]["doc_id"])
    print(msmarco_processed[idx]["original"])
    print("-" * 50)

DOC ID: 2
Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an
--------------------------------------------------
DOC ID: 7
Manhattan Project. The Manhattan Project was a research and development undertaking during World War
--------------------------------------------------
DOC ID: 3
The Manhattan Project was the name for a project conducted during World War II, to develop the first
--------------------------------------------------
DOC ID: 8
In June 1942, the United States Army Corps of Engineersbegan the Manhattan Project- The secret name 
--------------------------------------------------
DOC ID: 1
The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peacefu
--------------------------------------------------


In [53]:
from rank_bm25 import BM25Okapi

In [54]:
tokenized_docs = []

for doc in msmarco_processed:
    tokenized_docs.append(
        doc["processed"]
    )

print(tokenized_docs[0][:10])

['presence', 'communication', 'amid', 'scientific', 'mind', 'equally', 'important', 'success', 'manhattan', 'project']


In [55]:
bm25 = BM25Okapi(tokenized_docs)

print("BM25 Ready")

BM25 Ready


In [56]:
def bm25_search(query, top_k=5):

    query_tokens = preprocess_text(query)

    scores = bm25.get_scores(
        query_tokens
    )

    top_indices = scores.argsort()[-top_k:][::-1]

    return top_indices

In [57]:
results = bm25_search(
    "manhattan project"
)

print(results)

[2 7 3 8 6]


In [58]:
for idx in results:
    print("DOC ID:", msmarco_processed[idx]["doc_id"])
    print(msmarco_processed[idx]["original"])
    print("-" * 50)

DOC ID: 2
Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an
--------------------------------------------------
DOC ID: 7
Manhattan Project. The Manhattan Project was a research and development undertaking during World War
--------------------------------------------------
DOC ID: 3
The Manhattan Project was the name for a project conducted during World War II, to develop the first
--------------------------------------------------
DOC ID: 8
In June 1942, the United States Army Corps of Engineersbegan the Manhattan Project- The secret name 
--------------------------------------------------
DOC ID: 6
Nor will it attempt to substitute for the extraordinarily rich literature on the atomic bombs and th
--------------------------------------------------


In [59]:
print(results)

[2 7 3 8 6]


In [60]:
from sentence_transformers import SentenceTransformer

In [61]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

print("Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Loaded


In [62]:
documents_text = []

for doc in msmarco_processed:
    documents_text.append(
        " ".join(doc["processed"])
    )

print(len(documents_text))

1000


In [63]:
doc_embeddings = model.encode(
    documents_text,
    show_progress_bar=True
)

print(doc_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)


In [64]:
print(doc_embeddings.shape)

(1000, 384)


In [65]:
print(doc_embeddings.shape)

(1000, 384)


In [66]:
doc_embeddings.shape

(1000, 384)

In [68]:
import pickle
from scipy import sparse

pickle.dump(vectorizer, open("/content/drive/MyDrive/IR_Project/tfidf_vectorizer.pkl", "wb"))
sparse.save_npz("/content/drive/MyDrive/IR_Project/tfidf_matrix.npz", tfidf_matrix)

In [69]:
import pickle

pickle.dump(bm25, open("/content/drive/MyDrive/IR_Project/bm25.pkl", "wb"))

In [70]:
import numpy as np

np.save("/content/drive/MyDrive/IR_Project/doc_embeddings.npy", doc_embeddings)

In [71]:
import pickle

pickle.dump(msmarco_processed,
            open("/content/drive/MyDrive/IR_Project/msmarco_processed.pkl", "wb"))

In [72]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
def hybrid_parallel_search(query, top_k=5, alpha=0.5):

    # 1. preprocess
    query_tokens = preprocess_text(query)
    query_text = " ".join(query_tokens)

    # 2. BM25 scores
    bm25_scores = bm25.get_scores(query_tokens)

    # 3. Embedding query
    query_vec = model.encode([query_text])
    emb_scores = cosine_similarity(query_vec, doc_embeddings)[0]

    # 4. Normalize (مهم جداً)
    bm25_norm = (bm25_scores - np.min(bm25_scores)) / (np.max(bm25_scores) - np.min(bm25_scores) + 1e-9)
    emb_norm = (emb_scores - np.min(emb_scores)) / (np.max(emb_scores) - np.min(emb_scores) + 1e-9)

    # 5. Combine
    final_scores = alpha * bm25_norm + (1 - alpha) * emb_norm

    # 6. Top results
    top_indices = np.argsort(final_scores)[-top_k:][::-1]

    return top_indices

In [74]:
results = hybrid_parallel_search("manhattan project")
print(results)

[2 7 3 8 6]


In [75]:
for idx in results:
    print(msmarco_processed[idx]["doc_id"])
    print(msmarco_processed[idx]["original"])
    print("-"*50)

2
Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an
--------------------------------------------------
7
Manhattan Project. The Manhattan Project was a research and development undertaking during World War
--------------------------------------------------
3
The Manhattan Project was the name for a project conducted during World War II, to develop the first
--------------------------------------------------
8
In June 1942, the United States Army Corps of Engineersbegan the Manhattan Project- The secret name 
--------------------------------------------------
6
Nor will it attempt to substitute for the extraordinarily rich literature on the atomic bombs and th
--------------------------------------------------


In [76]:
def hybrid_serial_search(query, top_k=5, bm25_k=100):

    query_tokens = preprocess_text(query)
    query_text = " ".join(query_tokens)

    # 1. BM25 أول مرحلة
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top = np.argsort(bm25_scores)[-bm25_k:][::-1]

    # 2. Embedding rerank
    query_vec = model.encode([query_text])

    candidate_embeddings = doc_embeddings[bm25_top]

    emb_scores = cosine_similarity(query_vec, candidate_embeddings)[0]

    # 3. ترتيب نهائي
    top_local = np.argsort(emb_scores)[-top_k:][::-1]

    final_docs = bm25_top[top_local]

    return final_docs

In [77]:
results = hybrid_serial_search("manhattan project")
print(results)

[2 7 3 6 8]


In [80]:
!pip install gensim
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.0 MB/s eta 0:00:00


In [82]:
query = "manhattan project"

# 1. Preprocess query
query_tokens = preprocess_text(query)
query_text = " ".join(query_tokens)

# 2. Get TF-IDF scores for all documents
query_vector = vectorizer.transform([query_text])
tfidf_scores_raw = cosine_similarity(query_vector, tfidf_matrix)[0]

# 3. Get BM25 scores for all documents
bm25_scores_raw = bm25.get_scores(query_tokens)

# 4. Get Embedding scores for all documents
query_vec = model.encode([query_text])
emb_scores_raw = cosine_similarity(query_vec, doc_embeddings)[0]

# 5. Normalize scores (similar to hybrid_parallel_search)
epsilon = 1e-9 # Small value to prevent division by zero

tfidf_score = (tfidf_scores_raw - np.min(tfidf_scores_raw)) / \
              (np.max(tfidf_scores_raw) - np.min(tfidf_scores_raw) + epsilon)

bm25_score = (bm25_scores_raw - np.min(bm25_scores_raw)) / \
             (np.max(bm25_scores_raw) - np.min(bm25_scores_raw) + epsilon)

embedding_score = (emb_scores_raw - np.min(emb_scores_raw)) / \
                  (np.max(emb_scores_raw) - np.min(emb_scores_raw) + epsilon)

final_score = (
    0.3 * tfidf_score +
    0.3 * bm25_score +
    0.4 * embedding_score
)

In [83]:
def search(query, mode="serial"):
    if mode == "serial":
        return hybrid_serial_search(query)
    else:
        return hybrid_parallel_search(query)